# M01 学生带做Notebook｜工程数据质量与预处理

贯穿案例：装配式构件安装质检记录。

本Notebook完成一条最小证据链：**规则 → 问题记录 → 处理动作 → 理由 → 保存结果**。


## 1. 学习目标与运行方式

完成后你应能：

1. 读取原始记录、质量规则、显式映射和数据字典；
2. 解释R04与R06检查什么；
3. 选择一条规则并定位一条问题记录；
4. 写出处理动作和理由，保存为CSV。

按顺序运行全部代码单元。课堂只需要修改`FOCUS_RULE_ID`、选择行号并填写两段解释文字。


In [ ]:
from pathlib import Path
import platform

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for preferred_font in ["Microsoft YaHei", "Heiti SC", "SimHei", "Arial Unicode MS"]:
    if preferred_font in available_fonts:
        plt.rcParams["font.sans-serif"] = [preferred_font]
        break
plt.rcParams["axes.unicode_minus"] = False

from classroom_quality import (
    build_classroom_issue_log,
    duplicate_overview,
    issue_count_table,
    load_classroom_inputs,
    select_student_evidence,
    standardize_for_classroom,
)

BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("工作目录：课堂素材包根目录")

## 2. 数据、规则和映射分别回答什么

- 原始数据：记录了什么；
- 质量规则：怎样判断；
- 映射表：哪些写法可以有依据地统一；
- 数据字典：字段表示什么、采用什么类型和单位。

本案例一行代表一条安装检查记录；业务事件键为`component_id + inspection_round`。

In [ ]:
raw, rules, mapping, dictionary = load_classroom_inputs(BASE_DIR)

print(f"原始数据：{raw.shape[0]}行 × {raw.shape[1] - 1}列（source_row_number为课堂新增追踪列）")
print(f"质量规则：{len(rules)}条")
print(f"映射关系：{len(mapping)}条")

raw.head(3)


## 3. 五个基础质量维度

| 维度 | 核心问题 | 本案例规则示例 |
|---|---|---|
| 完整性 | 任务需要的值或记录是否存在 | R01 |
| 准确性 | 记录与可信事实是否一致 | 需要复测或可靠参考事实 |
| 一致性 | 格式、单位或字段关系是否相容 | R04、R06、R09、R11 |
| 及时性 | 是否在任务要求的时间内形成 | R05 |
| 唯一性 | 一个记录或业务事件是否只有一个有效表达 | R02、R03 |

值域或关系规则通过，说明记录满足该项规则；准确性仍需可信参考事实。


In [ ]:
focus_rule_table = rules.loc[
    rules["rule_id"].isin(["R04", "R06"]),
    ["rule_id", "quality_dimension", "field_scope", "rule_description_zh", "suggested_action", "rule_version"],
]

focus_dictionary = dictionary.loc[
    dictionary["field_name"].isin([
        "install_time", "inspection_time", "vertical_deviation_value", "deviation_unit"
    ]),
    ["field_name", "data_type", "unit", "description_zh", "role"],
]

display(focus_rule_table)
display(focus_dictionary)

## 4. 初步检查只负责定位

下面先统一特殊缺失码、解析显式时间格式、规范可识别的偏差单位，再查看重复概况。

重复概况不自动触发删除：完全重复、记录ID重复、业务重复和合法复检需要分别判断。


In [ ]:
standardized = standardize_for_classroom(raw, mapping)
duplicate_summary = duplicate_overview(standardized)
issue_log = build_classroom_issue_log(standardized, rules)
issue_counts = issue_count_table(issue_log)

display(duplicate_summary)
display(issue_counts)

assert raw.shape == (372, 18), "原始文件应为372行、17个发布字段，加1个追踪列"
assert duplicate_summary.set_index("检查项").loc["完全重复多余行", "数量"] == 6
assert duplicate_summary.set_index("检查项").loc["业务重复组", "数量"] == 6
assert issue_counts.set_index("rule_id").loc["R04", "issue_count"] == 16
assert issue_counts.set_index("rule_id").loc["R06", "issue_count"] == 5
print("课堂检查点通过：R04=16，R06=5")


In [ ]:
plot_data = issue_counts.set_index("rule_id")["issue_count"]
ax = plot_data.plot(
    kind="bar",
    color=["#2F5D8A", "#D07A32"],
    width=0.6,
    figsize=(7.2, 4.2),
    legend=False,
)
ax.set_title("课堂规则问题数量")
ax.set_xlabel("规则ID")
ax.set_ylabel("问题记录数")
ax.bar_label(ax.containers[0], padding=3)
plt.xticks(rotation=0)
plt.tight_layout()
figure_path = OUTPUT_DIR / "M12_规则问题数量.png"
plt.savefig(figure_path, dpi=160)
plt.show()
print("图表已保存：", figure_path)

## 5. R04与R06怎样形成问题记录

- **R04：**安装时间或检查时间无法解析，或者检查时间早于安装完成时间；
- **R06：**偏差单位无法按公开映射表规范为`mm`或`cm`。

问题清单保留规则ID、原始行号、相关字段、原始值、问题描述和参考动作。


In [ ]:
# 课堂只修改引号中的规则ID：可选"R04"或"R06"
FOCUS_RULE_ID = "R04"

if FOCUS_RULE_ID not in {"R04", "R06"}:
    raise ValueError('FOCUS_RULE_ID只能填写"R04"或"R06"')

selected_rule = focus_rule_table.loc[focus_rule_table["rule_id"].eq(FOCUS_RULE_ID)]
focus_issues = issue_log.loc[issue_log["rule_id"].eq(FOCUS_RULE_ID)].reset_index(drop=True)

display(selected_rule)
print(f"{FOCUS_RULE_ID}问题数：{len(focus_issues)}")
display(focus_issues.head(8))


## 6. 学生任务：解释一条问题记录

从上表中选择一条记录，完成四项证据：

1. 规则与触发条件；
2. 问题记录和原始行号；
3. 建议处理动作；
4. 采用该动作的理由。

`SELECTED_RESULT_ROW`从0开始计数。动作与理由要能让另一位同学依据规则或字段定义复查。


In [ ]:
# 选择上表中的一行，从0开始计数
SELECTED_RESULT_ROW = 0

# 在引号内填写你的解释
STUDENT_ACTION = "请在这里填写你的处理动作"
STUDENT_REASON = "请在这里填写你的理由，引用规则、字段定义或来源证据"

display(focus_issues.iloc[[SELECTED_RESULT_ROW]])


In [ ]:
student_evidence = select_student_evidence(
    issue_log=issue_log,
    focus_rule_id=FOCUS_RULE_ID,
    selected_result_row=SELECTED_RESULT_ROW,
    student_action=STUDENT_ACTION,
    student_reason=STUDENT_REASON,
)

result_path = OUTPUT_DIR / "M11_个人规则检查结果.csv"
student_evidence.to_csv(result_path, index=False, encoding="utf-8-sig")

display(student_evidence)
print("个人规则检查结果已保存：", result_path)


## 7. 处理前后比较

处理前后比较使用相同规则，同时报告适用数、通过数、失败数和通过率。R02检查记录ID唯一，R03检查业务事件唯一。

In [ ]:
summary_path = BASE_DIR / "data" / "M10_处理前后质量汇总.csv"
before_after = pd.read_csv(summary_path)
comparison = before_after.loc[
    before_after["rule_id"].isin(["R02", "R03"]),
    ["stage", "rule_id", "applicable_count", "pass_count", "fail_count", "pass_rate"],
].copy()
comparison["pass_rate"] = comparison["pass_rate"].map(lambda value: f"{value:.2%}")
display(comparison)

assert set(comparison["stage"]) == {"before", "after"}
assert set(comparison["rule_id"]) == {"R02", "R03"}

## 8. 检查与下一步

当堂保存：

- 当前Notebook；
- `outputs/M11_个人规则检查结果.csv`；
- `outputs/M12_规则问题数量.png`。

课后完成另一条规则，并从R02或R03中选择一条进行处理前后比较，写100—150字处理说明。

解释时依次说明用途和粒度、规则、问题性质、处理动作、处理前后结果和追踪依据。

In [ ]:
expected_files = [
    OUTPUT_DIR / "M11_个人规则检查结果.csv",
    OUTPUT_DIR / "M12_规则问题数量.png",
]
missing_files = [str(path) for path in expected_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(f"缺少输出文件：{missing_files}")

print("PASS：Notebook已从头运行，课堂输出文件均已生成。")
